# Mental Health RAG Chatbot

Pipeline: chunk reference doc → embed with `all-MiniLM-L6-v2` → FAISS → retrieve → generate with flan-t5-small.

Plug in any extra PDFs at `../docs/` (use PyPDF2 / pypdf to extract text first).

In [ ]:
import os, json, numpy as np
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
DOC = '../docs/mental_health_reference.md'
text = open(DOC, encoding='utf-8').read()
def chunk(t, size=400, overlap=80):
    out, i = [], 0
    while i < len(t):
        out.append(t[i:i+size]); i += size - overlap
    return [c.strip() for c in out if c.strip()]
chunks = chunk(text); len(chunks), chunks[0][:200]

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embs = model.encode(chunks, convert_to_numpy=True, normalize_embeddings=True)
embs.shape

In [ ]:
index = faiss.IndexFlatIP(embs.shape[1]); index.add(embs.astype(np.float32))
def retrieve(q, k=3):
    qv = model.encode([q], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    s, i = index.search(qv, k); return [(chunks[j], float(s[0][n])) for n, j in enumerate(i[0])]
retrieve('How do I cope with insomnia?')

## Generative answer (flan-t5)

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
tok = T5Tokenizer.from_pretrained('google/flan-t5-small')
llm = T5ForConditionalGeneration.from_pretrained('google/flan-t5-small')
def answer(q):
    ctx = '\n'.join(t for t,_ in retrieve(q,3))
    prompt = f'Answer using only this context:\n{ctx}\n\nQuestion: {q}\nAnswer:'
    out = llm.generate(**tok(prompt, return_tensors='pt', truncation=True, max_length=1024), max_new_tokens=160)
    return tok.decode(out[0], skip_special_tokens=True)
answer('What is the 4-7-8 breathing technique used for?')

In [ ]:
STORE = '../models/faiss_store'; os.makedirs(STORE, exist_ok=True)
faiss.write_index(index, f'{STORE}/index.faiss')
json.dump(chunks, open(f'{STORE}/chunks.json','w', encoding='utf-8'))
json.dump({'model':'sentence-transformers/all-MiniLM-L6-v2','dim':int(embs.shape[1])}, open(f'{STORE}/meta.json','w'))
print('saved.')